In [14]:
import os
import json
import pandas as pd

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

print("라이브러리 준비 완료")

라이브러리 준비 완료


In [4]:
history = pd.read_csv(
    "rag_data/synthetic_maintenance_history_30.csv"
)

In [5]:
print(history.shape)
print(history.head())

(30, 17)
  record_id equipment_id failure_type       failure_name   symptom  \
0    H-0001         M-02          TWF  Tool Wear Failure  공구 마모 증가   
1    H-0002         M-03          TWF  Tool Wear Failure  공구 마모 증가   
2    H-0003         M-04          TWF  Tool Wear Failure  공구 마모 증가   
3    H-0004         M-05          TWF  Tool Wear Failure  공구 마모 증가   
4    H-0005         M-06          TWF  Tool Wear Failure  공구 마모 증가   

   air_temperature_K  process_temperature_K  rotational_speed_rpm  torque_Nm  \
0              299.0                  309.0                  1450       42.0   
1              299.2                  309.2                  1485       44.1   
2              299.4                  309.4                  1520       46.2   
3              299.6                  309.6                  1555       48.3   
4              299.8                  309.8                  1590       50.4   

   tool_wear_min                                  inspection  \
0            185  공구 마모 상

In [7]:
import json

with open(
    "agent_input_sample_1391.json",
    "r",
    encoding="utf-8"
) as f:
    agent_input = json.load(f)

print(agent_input)


{'sample_id': 1391, 'machine_failure_probability': 0.94, 'risk_level': 'HIGH', 'top_features': [{'feature': 'Rotational speed [rpm]', 'value': 2737.0, 'shap_value': 0.2701}, {'feature': 'Torque [Nm]', 'value': 8.8, 'shap_value': 0.2691}, {'feature': 'Air temperature [K]', 'value': 298.9, 'shap_value': -0.0548}, {'feature': 'Tool wear [min]', 'value': 142.0, 'shap_value': -0.0448}, {'feature': 'Process temperature [K]', 'value': 310.2, 'shap_value': -0.0151}]}


In [9]:
query = """
고장 위험도 높음.
회전 속도가 매우 높고 토크가 매우 낮음.
구동부 부하 또는 동력 전달 계통 이상 가능성.
Power Failure, 동력 조건 이상 관련 유사 정비 사례 검색.
"""

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 검색 문서
history["search_text"] = (
    history["failure_type"].astype(str) + " " +
    history["failure_name"].astype(str) + " " +
    history["symptom"].astype(str) + " " +
    history["inspection"].astype(str) + " " +
    history["confirmed_cause"].astype(str) + " " +
    history["action_taken"].astype(str)
)

# 새 검색문
query = """
고장 위험도 높음
회전 속도 매우 높음
토크 매우 낮음
회전 속도와 토크 조합 이상
구동부 부하 상태 이상
동력 전달 계통 이상
Power Failure
PWF
"""

# 처음부터 다시 벡터화
vectorizer = TfidfVectorizer()

matrix = vectorizer.fit_transform(
    history["search_text"].tolist() + [query]
)

# 유사도 재계산
scores = cosine_similarity(
    matrix[-1],
    matrix[:-1]
)[0]

history["similarity"] = scores

# 상위 5개 다시 생성
top_cases = history.sort_values(
    "similarity",
    ascending=False
).head(5)

display(
    top_cases[
        [
            "record_id",
            "failure_type",
            "failure_name",
            "symptom",
            "inspection",
            "action_taken",
            "similarity"
        ]
    ]
)

,record_id,failure_type,failure_name,symptom,inspection,action_taken,similarity
15,H-0016,PWF,Power Failure,동력 조건 이상,회전 속도와 토크 조합 확인 / 구동부 부하 상태 확인 / 동력 전달 계통 이상 여...,구동부 부하 원인 점검,0.449330
12,H-0013,PWF,Power Failure,동력 조건 이상,회전 속도와 토크 조합 확인 / 구동부 부하 상태 확인 / 동력 전달 계통 이상 여...,구동부 부하 원인 점검,0.449330
13,H-0014,PWF,Power Failure,동력 조건 이상,회전 속도와 토크 조합 확인 / 구동부 부하 상태 확인 / 동력 전달 계통 이상 여...,이상 부품/결합 상태 확인,0.430278
16,H-0017,PWF,Power Failure,동력 조건 이상,회전 속도와 토크 조합 확인 / 구동부 부하 상태 확인 / 동력 전달 계통 이상 여...,이상 부품/결합 상태 확인,0.430278
17,H-0018,PWF,Power Failure,동력 조건 이상,회전 속도와 토크 조합 확인 / 구동부 부하 상태 확인 / 동력 전달 계통 이상 여...,정상 범위 확인 후 시험 가동,0.410399


In [15]:
history = pd.read_csv(
    "rag_data/synthetic_maintenance_history_30.csv"
)

documents = []

for _, row in history.iterrows():

    text = f"""
고장 유형: {row['failure_type']}
고장명: {row['failure_name']}
증상: {row['symptom']}

센서 상태:
- 회전 속도: {row['rotational_speed_rpm']} rpm
- 토크: {row['torque_Nm']} Nm
- 공구 마모: {row['tool_wear_min']} min
- 공기 온도: {row['air_temperature_K']} K
- 공정 온도: {row['process_temperature_K']} K

점검:
{row['inspection']}

확인 원인:
{row['confirmed_cause']}

조치:
{row['action_taken']}
"""

    documents.append(
        Document(
            page_content=text,
            metadata={
                "record_id": row["record_id"],
                "failure_type": row["failure_type"],
                "manual_section": row["manual_section"]
            }
        )
    )

print("Document 개수:", len(documents))

Document 개수: 30
